# Tutorial: TLV Diffusion Model

**Purpose:** Hands-on guide to the conditional diffusion model — the generative
companion to the TLV VAEs.
**Author:** Hallett Lab
**Date:** 2026-06-15

This tutorial uses the diffusion inference API to:
- **Generate** novel *Candida albicans* colonies from random semantic codes
- **Reconstruct** real colonies
- **Interpolate** between two colonies in semantic-code space
- **Encode** images to their semantic codes

The model (ported from Jose's *38-diffusion* / BW-1.0.0) is a pure-PyTorch
**conditional DDPM/DDIM**: a `SemanticEncoder` maps an image to a low-dimensional
semantic code, and a FiLM-conditioned `DiffusionUNet` denoises 128×128 images
conditioned on that code.

> **Prerequisites:** a diffusion checkpoint registered in the model zoo (architecture
> `diffusion_vae`) or available on disk, and the project environment (`uv sync`). A GPU
> is strongly recommended — reconstruction runs the full reverse diffusion chain.

## 1. Find a diffusion checkpoint

Diffusion checkpoints register in the model zoo under the architecture
`diffusion_vae`. If your zoo has none, point `CANDESCENCE_ZOO` at a populated zoo
or load a checkpoint directly by path (see the next cell).

In [ ]:
from candescence.core.model_zoo import ModelZoo
from candescence.tlv.diffusion import load_diffusion_model
from candescence.tlv.diffusion import inference as diff_inf

zoo = ModelZoo()
diffusion_models = [
    m for m in zoo.list_models(project="tlv")
    if m.architecture == "diffusion_vae" and m.exists()
]

for m in diffusion_models:
    print(f"{m.id:24s} -> {m.get_checkpoint_path()}")

if not diffusion_models:
    print("No diffusion checkpoint found in the zoo.")
    print("Load one directly by path instead, e.g.:")
    print("    loaded = load_diffusion_model('/path/to/ckpt_epoch_XXXX.pt')")

## 2. Load the model

`load_diffusion_model` returns an eval-ready `LoadedDiffusionModel` bundling the
network and its precomputed diffusion schedule. It prefers EMA weights when present
and handles both the flat BW-1.0.0 and the nested version-5 checkpoint config
layouts.

In [ ]:
import torch

if diffusion_models:
    ckpt = str(diffusion_models[0].get_checkpoint_path())
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    loaded = load_diffusion_model(ckpt, device=device)
    print(f"Loaded {diffusion_models[0].id}")
    print(f"  latent_dim     = {loaded.latent_dim}")
    print(f"  input_channels = {loaded.input_channels}")
    print(f"  image size     = {loaded.img_size}x{loaded.img_size}")
    print(f"  device         = {loaded.device}")
else:
    loaded = None
    print("Set `loaded = load_diffusion_model(<path>)` to continue.")

A small display helper used throughout. Inference functions return lists of
`[0, 1]` numpy arrays — grayscale `(H, W)` or RGB `(H, W, 3)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def show_images(images, titles=None, cols=4, figsize=None):
    """Display a list of [0, 1] numpy images (grayscale or RGB) in a grid."""
    n = len(images)
    cols = min(cols, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize or (2.4 * cols, 2.4 * rows),
                             squeeze=False)
    for i, ax in enumerate(axes.flat):
        if i < n:
            img = images[i]
            ax.imshow(img, cmap="gray" if img.ndim == 2 else None, vmin=0, vmax=1)
            if titles:
                ax.set_title(titles[i], fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def tensor_to_image(t):
    """(C, H, W) tensor in [-1, 1] -> [0, 1] numpy image for display."""
    t = ((t + 1) / 2).clamp(0, 1).cpu().numpy()
    return t[0] if t.shape[0] == 1 else t.transpose(1, 2, 0)

## 3. Generate novel colonies

With no `z_sem` argument, `generate` samples random semantic codes ~ N(0, I) and
denoises them with DDIM. Generation is fast; `ddim_steps` trades speed for quality
(fewer steps = faster, lower quality). Pass `seed` for reproducible samples.

In [ ]:
if loaded is not None:
    samples = diff_inf.generate(loaded, n=8, ddim_steps=30, seed=0)
    show_images(samples, titles=[f"sample {i}" for i in range(len(samples))])

## 4. Reconstruct real colonies

`reconstruct` encodes each image to its semantic code, noises it to the final
timestep, then DDIM-denoises conditioned on that code. This runs the **full reverse
chain**, so use a GPU. We load a few bundled sample images first.

In [ ]:
from pathlib import Path

import torchvision.transforms.functional as TF
from PIL import Image
import candescence

# Resolve the bundled sample images relative to the installed package, so this
# cell works regardless of the notebook's working directory.
REPO_ROOT = Path(candescence.__file__).resolve().parents[2]
SAMPLE_DIR = REPO_ROOT / "data" / "sample" / "images"


def load_image_tensor(path, channels, img_size):
    """Load one image as a (C, H, W) tensor normalised to [-1, 1]."""
    mode = "L" if channels == 1 else "RGB"
    im = Image.open(path).convert(mode).resize((img_size, img_size))
    return TF.to_tensor(im) * 2 - 1


if loaded is not None:
    sample_paths = sorted(SAMPLE_DIR.glob("*.png"))[:4]
    x = torch.stack([
        load_image_tensor(p, loaded.input_channels, loaded.img_size)
        for p in sample_paths
    ])

    recon = diff_inf.reconstruct(loaded, x, ddim_steps=30)
    originals = [tensor_to_image(t) for t in x]
    titles = ([f"orig {i}" for i in range(len(originals))]
              + [f"recon {i}" for i in range(len(recon))])
    show_images(originals + recon, titles=titles, cols=len(originals))

## 5. Interpolate between two colonies

`interpolate` encodes two images to their semantic codes, walks a straight line
between them, and generates an image at each step — a smooth morph through the
learned semantic space.

In [ ]:
if loaded is not None and len(sample_paths) >= 2:
    x_a = load_image_tensor(sample_paths[0], loaded.input_channels, loaded.img_size)
    x_b = load_image_tensor(sample_paths[1], loaded.input_channels, loaded.img_size)

    frames = diff_inf.interpolate(loaded, x_a, x_b, n_steps=8, ddim_steps=30)
    fracs = np.linspace(0, 1, len(frames))
    show_images(frames, titles=[f"{f:.2f}" for f in fracs], cols=len(frames))

## 6. The semantic code

`encode` returns the posterior means — the semantic codes the diffusion process is
conditioned on. These are the coordinates you interpolate through above, and a
compact representation you can cluster or relate to metadata.

In [ ]:
if loaded is not None:
    codes = diff_inf.encode(loaded, x)  # (B, latent_dim)
    print("semantic codes shape:", codes.shape)
    print("first code (first 8 dims):", np.round(codes[0][:8], 3))

## 7. Notes and further reading

- **DDIM steps** (`ddim_steps`): generation is fast and tolerates few steps;
  reconstruction quality improves with more, at the cost of time.
- **Device:** generation runs acceptably on CPU; reconstruction/interpolation
  really want a GPU.
- **In the app:** the same operations are available interactively on the
  **TLV Diffusion** page — `uv run streamlit run src/candescence/interface/app.py`.
  The model is currently inference-only and lives behind the sidebar **Research
  mode** toggle (it joins the public model tier in a later phase).

Further reading:
- Step-by-step doc: [`docs/tutorials/04_diffusion.md`](../docs/tutorials/04_diffusion.md)
- Architecture blueprint: [`docs/phase5_diffusion_blueprint.md`](../docs/phase5_diffusion_blueprint.md)
- The VAEs it complements: [`docs/tutorials/03_the_models.md`](../docs/tutorials/03_the_models.md)